# RGC Axon Targeting Data Analysis Workflow

This notebook analyzes RGC axon targeting data **after the dorsal-ventral percent positions have already been calculated**.

It does **not** manually annotate axons or calculate the percent position from raw image coordinates.

## Goal

Compare normalized axon entry positions across experimental groups.

Main measurement already present in the input CSV:

```text
normalized_DV_position
```

where:

```text
0%   = ventral-most optic tract fascicle
100% = dorsal-most optic tract fascicle
```

Example groups:

```text
control
nev_mutant
rescue
GFP_only
```

## Main biological question

```text
Do mutant or rescue conditions shift the D-V entry position of RGC axons in the tectum?
```

# 1. Required libraries and purpose

| Library | Purpose |
|---|---|
| `pathlib` | File and folder paths |
| `numpy` | Numeric calculations |
| `pandas` | Load, clean, and summarize CSV data |
| `matplotlib` | Plot figures |
| `seaborn` | Group comparison plots |
| `scipy.stats` | ANOVA and Kruskal-Wallis tests |
| `statsmodels` | Mixed-effects model accounting for embryo-level nesting |

Install if needed:

```bash
pip install numpy pandas matplotlib seaborn scipy statsmodels
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)

sns.set_context("notebook")
sns.set_style("whitegrid")

# 2. Input and output paths

Expected input CSV:

```text
../rgc_axon_targeting_project/data/axon_percent_positions.csv
```

Expected columns:

| Column | Meaning |
|---|---|
| `embryo_id` | larva/embryo identifier |
| `axon_id` | axon/arbor identifier |
| `group` | experimental group |
| `normalized_DV_position` | already-calculated D-V percent position |
| `include` | optional TRUE/FALSE inclusion flag |
| `notes` | optional notes |

Optional columns:

| Column | Meaning |
|---|---|
| `genotype` | genotype label |
| `construct` | electroporated construct |
| `image_file` | source image |
| `retina_position` | retinal source region |
| `tectum_region` | target tectum region |

In [ ]:
project_dir = Path("../rgc_axon_targeting_project")

data_dir = project_dir / "data"
output_dir = project_dir / "outputs"
plot_dir = output_dir / "plots"
table_dir = output_dir / "tables"

for folder in [output_dir, plot_dir, table_dir]:
    folder.mkdir(parents=True, exist_ok=True)

input_csv = data_dir / "axon_percent_positions.csv"

print("Input CSV:", input_csv)
print("Output directory:", output_dir)

# 3. Load axon percent-position data

This step imports the table where the D-V positions have already been measured/calculated.

No image annotation is performed here.

In [ ]:
df = pd.read_csv(input_csv)

print("Raw shape:", df.shape)
display(df.head())
print(df.columns.tolist())

# 4. Clean and validate data

This step:

1. Checks that required columns exist.
2. Converts `normalized_DV_position` to numeric.
3. Applies `include` filtering if present.
4. Flags out-of-range values below 0% or above 100%.

In [ ]:
required_cols = [
    "embryo_id",
    "axon_id",
    "group",
    "normalized_DV_position"
]

missing = [
    col for col in required_cols
    if col not in df.columns
]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["embryo_id"] = df["embryo_id"].astype(str)
df["axon_id"] = df["axon_id"].astype(str)
df["group"] = df["group"].astype(str)

df["normalized_DV_position"] = pd.to_numeric(
    df["normalized_DV_position"],
    errors="coerce"
)

if "include" in df.columns:
    df["include"] = (
        df["include"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y"])
    )

    included = df[df["include"]].copy()

else:
    included = df.copy()

included = included.dropna(
    subset=["normalized_DV_position"]
).copy()

included["out_of_range"] = (
    (included["normalized_DV_position"] < 0)
    | (included["normalized_DV_position"] > 100)
)

print("Included shape:", included.shape)

print("Out-of-range rows:")
display(included[included["out_of_range"]])

display(included.head())

# 5. Save cleaned analysis table

This is the main cleaned data table used for downstream analysis.

In [ ]:
clean_path = table_dir / "axon_percent_positions_clean.csv"

included.to_csv(
    clean_path,
    index=False
)

print("Saved:", clean_path)

# 6. Summary statistics by group

This summarizes:

```text
n axons
n embryos
mean position
median position
standard deviation
standard error
range
```

In [ ]:
group_summary = (
    included
    .groupby("group")
    .agg(
        n_axons=("axon_id", "count"),
        n_embryos=("embryo_id", "nunique"),
        mean_DV_position=("normalized_DV_position", "mean"),
        std_DV_position=("normalized_DV_position", "std"),
        median_DV_position=("normalized_DV_position", "median"),
        min_DV_position=("normalized_DV_position", "min"),
        max_DV_position=("normalized_DV_position", "max")
    )
    .reset_index()
)

group_summary["sem_DV_position"] = (
    group_summary["std_DV_position"]
    / np.sqrt(group_summary["n_axons"])
)

summary_path = table_dir / "group_summary_axon_percent_positions.csv"

group_summary.to_csv(
    summary_path,
    index=False
)

display(group_summary)

print("Saved:", summary_path)

# 7. Plot axon-level group comparison

This plot shows all measured axons.

Interpretation:

```text
Lower values = more ventral entry
Higher values = more dorsal entry
```

depending on the original anatomical convention used during measurement.

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=included,
    x="group",
    y="normalized_DV_position"
)

sns.stripplot(
    data=included,
    x="group",
    y="normalized_DV_position",
    color="black",
    alpha=0.5
)

plt.axhline(0, linestyle="--", color="gray")
plt.axhline(100, linestyle="--", color="gray")

plt.xlabel("Group")
plt.ylabel("Normalized D-V axon entry position (%)")
plt.title("Axon-Level D-V Entry Position by Group")
plt.tight_layout()

out_path = plot_dir / "axon_level_DV_position_by_group.tif"

plt.savefig(
    out_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", out_path)

# 8. Embryo-level aggregation

If several axons were measured from the same embryo, axons are not fully independent.

This step averages axons within embryo first:

```text
axon measurements → embryo mean
```

This prevents embryos with many axons from dominating the analysis.

In [ ]:
embryo_summary = (
    included
    .groupby(["embryo_id", "group"])
    .agg(
        mean_DV_position=("normalized_DV_position", "mean"),
        median_DV_position=("normalized_DV_position", "median"),
        n_axons=("axon_id", "count")
    )
    .reset_index()
)

embryo_summary_path = table_dir / "embryo_level_axon_percent_positions.csv"

embryo_summary.to_csv(
    embryo_summary_path,
    index=False
)

display(embryo_summary.head())

print("Saved:", embryo_summary_path)

# 9. Plot embryo-level group comparison

This is often the more conservative figure for reporting because each embryo contributes one mean value.

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=embryo_summary,
    x="group",
    y="mean_DV_position"
)

sns.stripplot(
    data=embryo_summary,
    x="group",
    y="mean_DV_position",
    color="black",
    alpha=0.6
)

plt.axhline(0, linestyle="--", color="gray")
plt.axhline(100, linestyle="--", color="gray")

plt.xlabel("Group")
plt.ylabel("Embryo mean D-V position (%)")
plt.title("Embryo-Level D-V Entry Position by Group")
plt.tight_layout()

out_path = plot_dir / "embryo_level_DV_position_by_group.tif"

plt.savefig(
    out_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", out_path)

# 10. Statistical tests at axon level

This performs:

```text
one-way ANOVA
Kruskal-Wallis test
```

These test whether groups differ in normalized D-V axon position.

Important caution:

If multiple axons come from the same embryo, axon-level tests may overestimate independence. Use embryo-level tests or mixed-effects models as stronger options.

In [ ]:
axon_groups = [
    group["normalized_DV_position"].dropna().values
    for _, group in included.groupby("group")
]

if len(axon_groups) >= 2:
    anova_stat, anova_p = stats.f_oneway(*axon_groups)
    kruskal_stat, kruskal_p = stats.kruskal(*axon_groups)

    axon_stats = pd.DataFrame([
        {
            "analysis_level": "axon",
            "test": "one_way_ANOVA",
            "statistic": anova_stat,
            "p_value": anova_p
        },
        {
            "analysis_level": "axon",
            "test": "Kruskal_Wallis",
            "statistic": kruskal_stat,
            "p_value": kruskal_p
        }
    ])

    axon_stats_path = table_dir / "axon_level_group_stat_tests.csv"

    axon_stats.to_csv(
        axon_stats_path,
        index=False
    )

    display(axon_stats)
    print("Saved:", axon_stats_path)

else:
    print("Need at least two groups for statistical testing.")

# 11. Statistical tests at embryo level

This compares embryo mean values across groups.

This is more conservative than axon-level analysis.

In [ ]:
embryo_groups = [
    group["mean_DV_position"].dropna().values
    for _, group in embryo_summary.groupby("group")
]

if len(embryo_groups) >= 2:
    anova_stat, anova_p = stats.f_oneway(*embryo_groups)
    kruskal_stat, kruskal_p = stats.kruskal(*embryo_groups)

    embryo_stats = pd.DataFrame([
        {
            "analysis_level": "embryo",
            "test": "one_way_ANOVA",
            "statistic": anova_stat,
            "p_value": anova_p
        },
        {
            "analysis_level": "embryo",
            "test": "Kruskal_Wallis",
            "statistic": kruskal_stat,
            "p_value": kruskal_p
        }
    ])

    embryo_stats_path = table_dir / "embryo_level_group_stat_tests.csv"

    embryo_stats.to_csv(
        embryo_stats_path,
        index=False
    )

    display(embryo_stats)
    print("Saved:", embryo_stats_path)

else:
    print("Need at least two groups for embryo-level testing.")

# 12. Mixed-effects model

This model keeps all axons but accounts for embryo nesting:

```text
normalized_DV_position ~ group + embryo_effect
```

Use this when there are multiple axons per embryo.

In [ ]:
try:
    mixed_model = smf.mixedlm(
        "normalized_DV_position ~ C(group)",
        data=included,
        groups=included["embryo_id"]
    ).fit()

    print(mixed_model.summary())

    mixed_path = table_dir / "mixed_effects_model_summary.txt"

    with open(mixed_path, "w") as f:
        f.write(mixed_model.summary().as_text())

    print("Saved:", mixed_path)

except Exception as e:
    print("Mixed-effects model could not be fit:")
    print(e)

# 13. Optional pairwise comparisons

If there are more than two groups, this performs pairwise Mann-Whitney U tests.

Example comparisons:

```text
control vs nev_mutant
control vs rescue
nev_mutant vs rescue
```

This is exploratory unless corrected for multiple comparisons.

In [ ]:
from itertools import combinations

pairwise_rows = []

groups_available = sorted(included["group"].dropna().unique())

for group_a, group_b in combinations(groups_available, 2):

    values_a = included.loc[
        included["group"] == group_a,
        "normalized_DV_position"
    ].dropna()

    values_b = included.loc[
        included["group"] == group_b,
        "normalized_DV_position"
    ].dropna()

    if len(values_a) > 0 and len(values_b) > 0:
        stat, p = stats.mannwhitneyu(
            values_a,
            values_b,
            alternative="two-sided"
        )

        pairwise_rows.append({
            "group_a": group_a,
            "group_b": group_b,
            "test": "Mann_Whitney_U",
            "statistic": stat,
            "p_value_uncorrected": p,
            "median_a": values_a.median(),
            "median_b": values_b.median(),
            "difference_median_b_minus_a": values_b.median() - values_a.median()
        })

pairwise_df = pd.DataFrame(pairwise_rows)

if len(pairwise_df) > 0:
    # Simple Bonferroni correction.
    pairwise_df["p_value_bonferroni"] = np.minimum(
        pairwise_df["p_value_uncorrected"] * len(pairwise_df),
        1.0
    )

    pairwise_path = table_dir / "pairwise_group_comparisons.csv"

    pairwise_df.to_csv(
        pairwise_path,
        index=False
    )

    display(pairwise_df)
    print("Saved:", pairwise_path)

else:
    print("No pairwise comparisons available.")

# 14. Final interpretation guide

The key output is:

```text
normalized_DV_position
```

Interpretation examples:

```text
control mean = 20%
nev_mutant mean = 55%
```

This would suggest that mutant axons enter the tectum at a more dorsal position than controls.

If rescue animals shift back toward the control distribution:

```text
rescue mean ≈ control mean
```

that supports rescue of normal axon pathfinding.

## Final deliverables

```text
1. cleaned axon percent-position CSV
2. group summary table
3. axon-level group plot
4. embryo-level group plot
5. axon-level statistics
6. embryo-level statistics
7. mixed-effects model summary
8. optional pairwise comparisons
```